# PACE-KG — Steps 1 → 4 (Colab Pipeline)

Run the cells **in order**. Each step saves its output as a JSON file and
offers a download button so you can keep intermediate results.

| Step | What it does | Output file |
|------|-------------|-------------|
| 1 | Marker PDF parsing + OCR fallback | `*_step1_parsed.json` |
| 2 | Markdown preprocessor (bucket typing, noise removal, cross-slide filter) | `*_step2_preprocessed.json` |
| 3 | SIFRank-style keyphrase extraction | `*_step3_keyphrases.json` |
| 4 | LLM triple extraction + 3-layer validation | `*_step4_triples.json` |

> **GPU recommended** for Step 3 (SBERT) and Step 4 (SBERT + LLM calls).  
> Runtime → Change runtime type → T4 GPU.

## Cell 0 — Install dependencies
Run once per Colab session.

In [1]:
# -- Core parsing deps --------------------------------------------------------
!pip install marker-pdf -q
!pip install pytesseract pillow pymupdf -q
!apt-get install -y tesseract-ocr -q

# -- NLP / ML deps ------------------------------------------------------------
!pip install sentence-transformers spacy -q
!python -m spacy download en_core_web_sm -q

# -- GLiNER (replaces SIFRank / spaCy noun chunks in Step 3) -----------------
!pip install gliner -q

# -- LLM / LangChain deps -----------------------------------------------------
!pip install langchain langchain-groq -q

# -- Sanity check -------------------------------------------------------------
import importlib, subprocess, sys
for pkg in ["marker", "pytesseract", "fitz", "PIL",
            "sentence_transformers", "spacy", "gliner", "langchain_groq"]:
    ok = importlib.util.find_spec(pkg) is not None
    print(f"{'OK' if ok else 'MISSING':6s}  {pkg}")

res = subprocess.run(["tesseract", "--version"], capture_output=True, text=True)
print(f"{'OK' if res.returncode == 0 else 'MISSING':6s}  tesseract",
      res.stdout.splitlines()[0] if res.returncode == 0 else "")

print("\nAll dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.

## Cell 1 — Upload PDF
Upload the lecture slide PDF you want to process.

In [2]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
DOC_ID   = Path(PDF_PATH).stem          # used as document identifier throughout
STEM     = DOC_ID                       # base name for all output files

print(f"Uploaded : {PDF_PATH}")
print(f"Doc ID   : {DOC_ID}")

Saving test3.pdf to test3.pdf
Uploaded : test3.pdf
Doc ID   : test3


## Step 1 — Marker PDF Parsing

Uses **Marker** (deep-learning PDF parser) to convert each slide into structured
markdown. For slides where Marker finds no text (image-only), a Tesseract OCR
fallback renders the page at 3× zoom and recovers text.

**Output:** `SlideMarkdown` per page — `slide_id`, `page_number`, `raw_markdown`, `doc_id`.

> First run downloads ~4 GB of Marker model weights — allow ~5 minutes.

In [3]:
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List

import fitz          # PyMuPDF
import pytesseract
from PIL import Image

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

# ── Config ────────────────────────────────────────────────────────────────
PAGE_SEP = "-" * 48             # Marker page separator (must match config)
# Note: PDF_PATH, DOC_ID, and STEM are already defined in the previous cell.

# ── Data model ────────────────────────────────────────────────────────────
@dataclass
class SlideMarkdown:
    slide_id:     str
    page_number:  int
    raw_markdown: str
    doc_id:       str

# ── Helper Functions ──────────────────────────────────────────────────────
def _is_empty_page(markdown: str) -> bool:
    """True when Marker found no real text on the page."""
    cleaned = markdown.strip()
    if cleaned in ["{0}", ""]:
        return True
    # Strip out image markdown tags and basic markdown formatting characters
    text_only = re.sub(r"!\[.*?\]\(.*?\)", "", cleaned).strip()
    text_only = re.sub(r"[#>\-\*\|]", "", text_only).strip()
    return len(text_only) < 10

def _ocr_page(doc: fitz.Document, page_number: int) -> str:
    """Render full page at 3× zoom and run Tesseract on it."""
    # fitz is 0-indexed, so we subtract 1 from the actual page_number
    page = doc[page_number - 1]
    mat  = fitz.Matrix(3, 3)             # 3× zoom for accuracy
    pix  = page.get_pixmap(matrix=mat)
    img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img, config="--psm 6")
    return text.strip()

# ── Main Pipeline ─────────────────────────────────────────────────────────

# 1. Open the document ONCE to get the absolute truth of its length
print(f"Opening '{PDF_PATH}' to establish actual page count...")
doc = fitz.open(PDF_PATH)
actual_page_count = len(doc)
print(f"PDF structurally contains exactly {actual_page_count} pages.")

# 2. Run Marker
print("Loading Marker models (first run downloads ~4 GB — allow ~5 min) ...")
model_dict = create_model_dict()
converter  = PdfConverter(
    artifact_dict=model_dict,
    config={"paginate_output": True, "page_separator": PAGE_SEP},
)

rendered      = converter(PDF_PATH)
full_markdown = rendered.markdown

# 3. Split the text, stripping whitespace from each chunk
raw_pages = [p.strip() for p in full_markdown.split(PAGE_SEP)]

# ── Build SlideMarkdown list ──────────────────────────────────────────────
slides_md: List[SlideMarkdown] = []
ocr_count = 0

# 4. Iterate strictly based on the PDF's ACTUAL page count
for idx in range(1, actual_page_count + 1):

    # Safely grab the Marker chunk. If Marker spit out fewer chunks
    # than actual pages for some reason, default to an empty string.
    if (idx - 1) < len(raw_pages):
        page_md = raw_pages[idx - 1].strip()
    else:
        page_md = ""

    # Check if the chunk is empty and trigger OCR fallback
    if _is_empty_page(page_md):
        print(f"  [slide_{idx:03d}] Marker: no text — running OCR ...")
        ocr_text = _ocr_page(doc, idx)

        if ocr_text:
            # Prefix each OCR line with '> OCR:' so downstream steps can detect it
            page_md = "\n".join(
                f"> OCR: {line}"
                for line in ocr_text.splitlines()
                if line.strip()
            )
            print(f"  [slide_{idx:03d}] OCR recovered {len(ocr_text):,} chars")
            ocr_count += 1
        else:
            print(f"  [slide_{idx:03d}] OCR found nothing — slide may be decorative, skipping")
            page_md = ""

    # Collapse excessive blank lines
    page_md = re.sub(r"\n{3,}", "\n\n", page_md)

    # Skip adding to the final payload if it's completely empty
    if not page_md:
        continue

    slides_md.append(SlideMarkdown(
        slide_id    = f"slide_{idx:03d}",
        page_number = idx,
        raw_markdown= page_md,
        doc_id      = DOC_ID,
    ))

# 5. Clean up by closing the PyMuPDF document
doc.close()

print(f"\nStep 1 complete: {len(slides_md)} slides ({ocr_count} used OCR fallback)")

# ── Save intermediate result ──────────────────────────────────────────────
STEP1_FILE = f"{STEM}_step1_parsed.json"
with open(STEP1_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slides_md], f, ensure_ascii=False, indent=2)
print(f"Saved: {STEP1_FILE} ({Path(STEP1_FILE).stat().st_size:,} bytes)")

Opening 'test3.pdf' to establish actual page count...
PDF structurally contains exactly 28 pages.
Loading Marker models (first run downloads ~4 GB — allow ~5 min) ...



































































































Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 32.02it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]
Detecting bboxes: 0it [00:00, ?it/s]


  [slide_001] Marker: no text — running OCR ...
  [slide_001] OCR recovered 142 chars
  [slide_028] Marker: no text — running OCR ...
  [slide_028] OCR recovered 395 chars

Step 1 complete: 28 slides (2 used OCR fallback)
Saved: test3_step1_parsed.json (15,348 bytes)


In [4]:
# ── Step 1 inspection ─────────────────────────────────────────────────────
# Change SHOW_SLIDES and SHOW_FULL to inspect different pages.
SHOW_SLIDES = [1, 2, 3]   # page numbers to preview
SHOW_FULL   = False        # True = full markdown, False = first 500 chars

from IPython.display import Markdown, display

print(f"Total slides: {len(slides_md)}\n")
for slide in slides_md:
    if slide.page_number not in SHOW_SLIDES:
        continue
    display(Markdown(f"---\n### Slide {slide.page_number} — `{slide.slide_id}`\n---"))
    md = slide.raw_markdown
    preview = md if SHOW_FULL else (md[:500] + ("..." if len(md) > 500 else ""))
    print(preview)

Total slides: 28



---
### Slide 1 — `slide_001`
---

> OCR: UNIVERSITY OF WESTMINSTER
> OCR: Zee :o il | =
> OCR: vvv
> OCR: 5COSC001W — Object Oriented Programming
> OCR: Week 5
> OCR: Dr. Barbara Villarini
> OCR: b.villarini@westminster.ac.uk


---
### Slide 2 — `slide_002`
---

![](_page_0_Picture_0.jpeg)

# **5COSC001W – Object Oriented Programming Week 5**

Dr. Barbara Villarini

b.villarini@westminster.ac.uk

{1}


---
### Slide 3 — `slide_003`
---

![](_page_1_Picture_0.jpeg)

# **Summary**

- Collections and Data Structure
- Arrays
- List, Queue, Map
- Searching and Sorting
- Linked List

{2}


In [5]:
# ── Download Step 1 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP1_FILE)
print(f"Downloading {STEP1_FILE} ...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 2 — Markdown Preprocessor

Four stages applied to the raw markdown from Step 1:

1. **Structural parsing** — lines routed into typed buckets: headings, bullets, table cells, captions, body text
2. **Noise removal** — page numbers, copyright lines, URLs, image tags discarded
3. **Cross-slide repetition filter** — blocks appearing on >50% of slides removed (recurring headers/footers)
4. **Assembly** — buckets joined into `clean_text`; `heading_phrases` stored for Step 3 boost

OCR-recovered lines (prefixed `> OCR:`) are re-classified into proper buckets instead of being dumped into captions.

**Output:** `SlideContent` per slide.

In [6]:
import re
from collections import Counter
from dataclasses import asdict, dataclass, field
from typing import List

# -- Data model ---------------------------------------------------------------
@dataclass
class SlideContent:
    slide_id:        str
    page_number:     int
    doc_id:          str
    headings:        List[str] = field(default_factory=list)
    body_text:       List[str] = field(default_factory=list)
    bullets:         List[str] = field(default_factory=list)
    table_cells:     List[str] = field(default_factory=list)
    captions:        List[str] = field(default_factory=list)
    code_lines:      List[str] = field(default_factory=list)
    heading_phrases: List[str] = field(default_factory=list)
    clean_text:      str       = ""
    ocr_applied:     bool      = False

# -- Noise patterns -----------------------------------------------------------
_NOISE_PATTERNS = [
    re.compile(r"^\s*page\s+\d+\s*(of\s+\d+)?\s*$", re.IGNORECASE),
    re.compile(r"^\s*\d+\s*$"),
    re.compile(r"^[cC].*"),                        # copyright lines
    re.compile(r"^\s*(references|bibliography)\s*$", re.IGNORECASE),
    re.compile(r"https?://\S+"),
    re.compile(r"^\s*\[\d+\]"),
    re.compile(r"^!\[.*?\]\(.*?\)$"),              # image refs ![]()
    re.compile(r"^quiz\s*:", re.IGNORECASE),        # quiz prompts
    re.compile(r"^\{\d+\}$"),                       # FIX 2: {N} slide number tags
    re.compile(
        r"^(input|output|legend|infrastructure)(\s+replaced.*|\s+kept.*|\s+storage.*)?$",
        re.IGNORECASE,
    ),
]

def _is_noise(text: str) -> bool:
    t = text.strip()
    return bool(t) and any(p.match(t) for p in _NOISE_PATTERNS)

# -- Code-line detector -------------------------------------------------------
def _is_code_line(text: str) -> bool:
    t = text.strip()
    if not t:
        return False
    if t.startswith("```"):
        return True
    code_chars = set("{}();=<>[]")
    code_char_count = sum(1 for c in t if c in code_chars)
    if len(t) > 5 and code_char_count / len(t) > 0.08:   # FIX 3: was 0.10
        return True
    if re.search(r"\b[a-z]+[A-Z][a-zA-Z]+\b", t):        # camelCase
        return True
    if re.search(r"\b[a-z]+_[a-z]+\b", t):                # snake_case
        return True
    if re.search(r"\s(==|!=|<=|>=|&&|\|\||:=|->|=>)\s", t):
        return True
    if t.endswith((";", "{", "}")):
        return True
    if re.match(r"^\s*(//|#\s|--|/\*|\*\s)", t):
        return True
    if re.search(r"\b\w{2,}\(", t) and code_char_count >= 2:
        return True
    if re.match(r"^(default\s+)?constructor\s+\w+\s*\(", t, re.IGNORECASE):
        return True
    if re.match(r"^__\w+__\s*\(", t):
        return True
    return False

# -- Inline markdown stripper -------------------------------------------------
_INLINE_MD = re.compile(
    r"(\*\*|__)(.*?)\1|(\*|_)(.*?)\3|`([^`]+)`|~~(.*?)~~|\[([^\]]+)\]\([^)]+\)",
    re.DOTALL,
)

def _strip_inline(text: str) -> str:
    prev = None
    while prev != text:
        prev = text
        text = _INLINE_MD.sub(
            lambda m: next(g for g in m.groups() if g is not None
                           and g not in ("**", "__", "*", "_")),
            text,
        )
    return text.strip()

# -- OCR noise detector -------------------------------------------------------
def _is_ocr_noise(text: str) -> bool:
    t = text.strip()
    if not t or len(t) < 4:
        return True
    words = t.split()
    if len(words) == 1 and not words[0].isalpha():
        return True
    non_ascii = sum(1 for c in t if ord(c) > 127)
    if non_ascii / len(t) > 0.1:
        return True
    letters = sum(1 for c in t if c.isalpha())
    if len(t) > 5 and letters / len(t) < 0.4:
        return True
    return False

# -- OCR line re-classifier ---------------------------------------------------
def _classify_ocr_line(text: str, sc) -> None:
    stripped = text.strip()
    if not stripped or _is_noise(stripped) or _is_ocr_noise(stripped):
        return
    if _is_code_line(stripped):
        sc.code_lines.append(stripped)
        return
    if stripped.startswith("#"):
        inner = re.sub(r"^#+\s*", "", stripped).strip()
        if inner:
            sc.headings.append(inner)
            sc.heading_phrases.append(inner)
    elif re.match(r"^[-*+]\s+", stripped):
        inner = re.sub(r"^[-*+]\s+", "", stripped).strip()
        # Also strip any leading Unicode geometric bullet marker after the ASCII bullet
        inner = BULLET_REGEX.sub("", inner).strip()
        if inner:
            sc.bullets.append(inner)
    elif stripped.startswith("|") and stripped.endswith("|"):
        if not re.match(r"^\|[\s\-\|]+\|$", stripped):
            cells = [c.strip() for c in stripped.split("|") if c.strip()]
            sc.table_cells.extend(cells)
    else:
        sc.body_text.append(stripped)

# -- FIX 1: Beamer/LaTeX bullet markers + full Unicode geometric shapes ------
# Covers standard markdown bullets (- * +), common Beamer Unicode symbols
# (U+2022 bullet, U+25A0-25FF geometric shapes, U+2700-27BF dingbats)
# used in academic PDFs as slide bullet markers (e.g. ◮ ▪ ► ▶ ➤)
BULLET_REGEX = re.compile(
    r"^\s*[-*+]\s*(?:[\u2022\u25A0-\u25FF\u2700-\u27BF]\s*)?|"
    r"^\s*[\u2022\u25A0-\u25FF\u2700-\u27BF]\s+"
)
_BEAMER_BULLET_RE = re.compile(
    r"^[\u2022\u25A0-\u25FF\u2700-\u27BF]\s*"
)

# -- Stage 1+2: parse single slide -------------------------------------------
def _parse_slide(slide_md) -> SlideContent:
    sc = SlideContent(
        slide_id    = slide_md.slide_id,
        page_number = slide_md.page_number,
        doc_id      = slide_md.doc_id,
    )
    for line in slide_md.raw_markdown.splitlines():
        s = line.strip()
        if not s:
            continue

        # Headings
        if s.startswith("#"):
            text = _strip_inline(re.sub(r"^#+\s*", "", s))
            if text and not _is_noise(text):
                sc.headings.append(text)
                sc.heading_phrases.append(text)

        # Standard markdown bullets (- or *)
        elif re.match(r"^[-*]\s+", s) and not s.startswith(("**", "*a", "*b", "*c")):
            m    = re.match(r"^[-*]\s+(.+)", s)
            text = _strip_inline(m.group(1)) if m else ""
            if text and not _is_noise(text):
                if _is_code_line(text):
                    sc.code_lines.append(text)
                else:
                    sc.bullets.append(text)

        # FIX 1: Beamer / Unicode geometric shape bullet markers
        elif _BEAMER_BULLET_RE.match(s):
            text = _strip_inline(BULLET_REGEX.sub("", s)).strip()
            # strip nested sub-bullet marker if present
            text = BULLET_REGEX.sub("", text).strip()
            if text and not _is_noise(text):
                if _is_code_line(text):
                    sc.code_lines.append(text)
                else:
                    sc.bullets.append(text)

        # Table rows
        elif s.startswith("|") and s.endswith("|"):
            cells = [_strip_inline(c.strip()) for c in s.split("|") if c.strip()]
            if all(re.fullmatch(r"[-:]+", c) for c in cells):
                continue
            sc.table_cells.extend(c for c in cells if c and not _is_noise(c))

        # Blockquotes: real captions OR OCR-recovered lines
        elif s.startswith(">"):
            inner = re.sub(r"^>\s*", "", s).strip()
            if inner.startswith("OCR:"):
                sc.ocr_applied = True
                _classify_ocr_line(inner[4:].strip(), sc)
            else:
                text = _strip_inline(inner)
                if text and not _is_noise(text):
                    sc.captions.append(text)

        # Body text
        else:
            text = _strip_inline(s)
            if not text or _is_noise(text):
                continue
            if _is_code_line(text):
                sc.code_lines.append(text)
            else:
                sc.body_text.append(text)

    return sc

# -- Stage 3: cross-slide repetition filter ----------------------------------
def _cross_slide_filter(contents: List[SlideContent]) -> List[SlideContent]:
    total = len(contents)
    if total < 5:
        print(f"Cross-slide filter skipped (only {total} slides -- need 5+)")
        return contents
    threshold = total * 0.5
    counts: Counter = Counter()
    for sc in contents:
        seen = set()
        for bucket in (sc.headings, sc.body_text, sc.bullets, sc.table_cells, sc.captions):
            for t in bucket:
                if t not in seen:
                    counts[t] += 1
                    seen.add(t)
    noise = {t for t, c in counts.items() if c > threshold}
    if noise:
        print(f"Cross-slide filter removed {len(noise)} repeated block(s):")
        for n in sorted(noise):
            print(f"  '{n[:80]}'")
    else:
        print("Cross-slide filter: nothing removed")
    for sc in contents:
        sc.headings    = [t for t in sc.headings    if t not in noise]
        sc.body_text   = [t for t in sc.body_text   if t not in noise]
        sc.bullets     = [t for t in sc.bullets     if t not in noise]
        sc.table_cells = [t for t in sc.table_cells if t not in noise]
        sc.captions    = [t for t in sc.captions    if t not in noise]
    return contents

# -- Stage 4: assemble clean_text --------------------------------------------
def _assemble(sc: SlideContent) -> None:
    parts = sc.headings + sc.body_text + sc.bullets + sc.table_cells + sc.captions
    sc.clean_text = " ".join(parts).strip()

# -- Run all stages ----------------------------------------------------------
print("Stage 1+2: parsing and noise removal ...")
slide_contents: List[SlideContent] = [_parse_slide(s) for s in slides_md]

print("Stage 3: cross-slide repetition filter ...")
slide_contents = _cross_slide_filter(slide_contents)

print("Stage 4: assembling clean_text ...")
for sc in slide_contents:
    _assemble(sc)

content_slides = [sc for sc in slide_contents if sc.clean_text.strip()]
print(f"\nStep 2 complete: {len(content_slides)}/{len(slide_contents)} slides have content")
print(f"  OCR applied on: {sum(1 for s in content_slides if s.ocr_applied)} slide(s)")

# Quick bucket check
beamer_bullet_slides = sum(1 for sc in content_slides if sc.bullets)
print(f"  Slides with bullets: {beamer_bullet_slides}")

STEP2_FILE = f"{STEM}_step2_preprocessed.json"
with open(STEP2_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slide_contents], f, ensure_ascii=False, indent=2)
print(f"Saved  : {STEP2_FILE} ({Path(STEP2_FILE).stat().st_size:,} bytes)")


Stage 1+2: parsing and noise removal ...
Stage 3: cross-slide repetition filter ...
Cross-slide filter: nothing removed
Stage 4: assembling clean_text ...

Step 2 complete: 28/28 slides have content
  OCR applied on: 2 slide(s)
  Slides with bullets: 18
Saved  : test3_step2_preprocessed.json (26,988 bytes)


In [7]:
# ── Step 2 inspection ─────────────────────────────────────────────────────
from IPython.display import Markdown, display

SHOW_SLIDES_2 = list(range(1, min(4, len(content_slides) + 1)))  # first 3 content slides

print(f"{'slide_id':<12} {'head':>4} {'body':>4} {'bull':>4} {'tbl':>4} {'cap':>4} {'code':>4} {'OCR':>4}  clean_text (first 120 chars)")
print("-" * 110)
for i, sc in enumerate(content_slides):
    flag = "Y" if sc.ocr_applied else "-"
    print(f"{sc.slide_id:<12} {len(sc.headings):>4} {len(sc.body_text):>4} "
          f"{len(sc.bullets):>4} {len(sc.table_cells):>4} {len(sc.captions):>4} "
          f"{len(sc.code_lines):>4} {flag:>4}  {sc.clean_text[:120]}")

print("\n" + "=" * 80)
print("Detailed view of first 3 content slides")
for sc in content_slides[:3]:
    display(Markdown(f"---\n### {sc.slide_id}  (page {sc.page_number})"))
    print(f"Headings    : {sc.headings}")
    print(f"Bullets     : {sc.bullets[:4]}")
    print(f"Body        : {sc.body_text[:2]}")
    print(f"Table cells : {sc.table_cells[:3]}")
    print(f"Captions    : {sc.captions[:2]}")
    print(f"Code lines  : {sc.code_lines[:2]}")
    print(f"OCR applied : {sc.ocr_applied}")
    print(f"clean_text  : {sc.clean_text[:300]}")

slide_id     head body bull  tbl  cap code  OCR  clean_text (first 120 chars)
--------------------------------------------------------------------------------------------------------------
slide_001       0    5    0    0    0    0    Y  UNIVERSITY OF WESTMINSTER Zee :o il | = 5COSC001W — Object Oriented Programming Week 5 Dr. Barbara Villarini
slide_002       1    2    0    0    0    0    -  5COSC001W – Object Oriented Programming Week 5 Dr. Barbara Villarini b.villarini@westminster.ac.uk
slide_003       1    0    4    0    0    0    -  Summary Arrays List, Queue, Map Searching and Sorting Linked List
slide_004       1    0    4    0    0    0    -  Recap on Arrays Arrays are the fundamental mechanism for collecting multiple values. An array is an ordered list o value
slide_005       0    2    2    0    0    0    -  Arrays scores[2] Refers to the value 85 (the 3rd value in the array) A particular value in an array is referenced using 
slide_006       1    0    5    0    0    0    -  A

---
### slide_001  (page 1)

Headings    : []
Bullets     : []
Body        : ['UNIVERSITY OF WESTMINSTER', 'Zee :o il | =']
Table cells : []
Captions    : []
Code lines  : []
OCR applied : True
clean_text  : UNIVERSITY OF WESTMINSTER Zee :o il | = 5COSC001W — Object Oriented Programming Week 5 Dr. Barbara Villarini


---
### slide_002  (page 2)

Headings    : ['5COSC001W – Object Oriented Programming Week 5']
Bullets     : []
Body        : ['Dr. Barbara Villarini', 'b.villarini@westminster.ac.uk']
Table cells : []
Captions    : []
Code lines  : []
OCR applied : False
clean_text  : 5COSC001W – Object Oriented Programming Week 5 Dr. Barbara Villarini b.villarini@westminster.ac.uk


---
### slide_003  (page 3)

Headings    : ['Summary']
Bullets     : ['Arrays', 'List, Queue, Map', 'Searching and Sorting', 'Linked List']
Body        : []
Table cells : []
Captions    : []
Code lines  : []
OCR applied : False
clean_text  : Summary Arrays List, Queue, Map Searching and Sorting Linked List


In [8]:
# ── Download Step 2 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP2_FILE)
print(f"Downloading {STEP2_FILE} ...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 3 — Keyphrase Extraction (SIFRank-style)

Implements the **SIFRankSqueezeBERT** algorithm using:
- **spaCy `en_core_web_sm`** — candidate noun-chunk extraction and linguistic validation
- **`all-MiniLM-L6-v2`** (~80 MB) — phrase and document embeddings

Adaptive filter pipeline (in order):
1. Extract up to 30 noun-chunk candidates (code lines excluded)
2. Score by cosine similarity to document embedding
3. Quality threshold ≥ 0.3
4. spaCy linguistic filter (has noun, not all-stop, length ≥ 3)
5. Noun-chunk cross-validation
6. Assign source type + heading boost (+0.20)
7. Near-duplicate deduplication via SBERT (sim ≥ 0.85)

**Output:** `Keyphrase` list per slide — `phrase`, `score`, `source_type`, `appears_in`.

In [9]:
import json, re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Dict

import spacy
from gliner import GLiNER
from sentence_transformers import SentenceTransformer, util as st_util

# -- Config -------------------------------------------------------------------
GLINER_MODEL           = "urchade/gliner_large-v2.1"  # full large on T4 GPU
GLINER_THRESHOLD       = 0.35   # lower = more recall; large model is precise enough
DEDUP_SIM_THRESHOLD    = 0.85   # SBERT cosine threshold for near-duplicate removal
KEYPHRASE_MAX          = 25     # max keyphrases per slide
HEADING_BOOST          = 0.15   # score bonus for heading-sourced phrases

# -- Entity labels ------------------------------------------------------------
# Generalized to universal academic concepts so the pipeline works for any
# subject domain (not just Computer Science).
# Anything else (professor names, university, slide numbers, dates) is ignored.
GLINER_LABELS = [
    "Academic Concept",
    "Theoretical Principle",
    "Technical Term",
    "Process or Method",
    "System or Framework",
    "Formula or Equation",
]
ENTITY_LABELS = GLINER_LABELS   # alias used in gliner.predict_entities calls

# -- Data model ---------------------------------------------------------------
@dataclass
class Keyphrase:
    phrase:      str
    score:       float
    source_type: str    # heading | bullet | table | caption | body
    slide_id:    str
    doc_id:      str
    appears_in:  str    # sentence containing the phrase

# -- Load models --------------------------------------------------------------
print("Loading GLiNER large-v2.1 on GPU ...")
gliner = GLiNER.from_pretrained(GLINER_MODEL)
print("GLiNER ready.")

print("Loading spaCy en_core_web_sm (sentence segmentation) ...")
nlp = spacy.load("en_core_web_sm")

print("Loading all-MiniLM-L6-v2 (deduplication) ...")
minilm = SentenceTransformer("all-MiniLM-L6-v2")
print("All models ready.")

# -- Helpers ------------------------------------------------------------------
def _assign_source_type(phrase: str, sc) -> str:
    pl = phrase.lower()
    for h in sc.headings:
        if pl in h.lower(): return "heading"
    for b in sc.bullets:
        if pl in b.lower(): return "bullet"
    for t in sc.table_cells:
        if pl in t.lower(): return "table"
    for c in sc.captions:
        if pl in c.lower(): return "caption"
    return "body"

def _find_sentence(phrase: str, clean_text: str) -> str:
    doc = nlp(clean_text)
    pl  = phrase.lower()
    for sent in doc.sents:
        if pl in sent.text.lower():
            return sent.text.strip()
    return clean_text[:200]

def _deduplicate(kps: List[Keyphrase]) -> List[Keyphrase]:
    if len(kps) <= 1:
        return kps
    sorted_kps = sorted(kps, key=lambda k: k.score, reverse=True)
    kept: List[Keyphrase] = []
    for kp in sorted_kps:
        emb = minilm.encode(kp.phrase, convert_to_tensor=True)
        is_dup = any(
            float(st_util.cos_sim(
                emb,
                minilm.encode(k.phrase, convert_to_tensor=True)
            )) >= DEDUP_SIM_THRESHOLD
            for k in kept
        )
        if not is_dup:
            kept.append(kp)
    return kept

# -- Main extraction ----------------------------------------------------------
def extract_keyphrases(sc) -> List[Keyphrase]:
    if not sc.clean_text.strip():
        return []

    # Build input text: headings first so GLiNER weights them naturally,
    # code_lines excluded -- GLiNER would otherwise extract variable names
    heading_text = " ".join(sc.headings)
    rest_text    = " ".join(
        sc.body_text + sc.bullets + sc.table_cells + sc.captions
    )
    extract_text = (heading_text + ". " + rest_text).strip() \
                   if heading_text and rest_text else (heading_text or rest_text)

    if not extract_text.strip():
        return []

    # GLiNER inference
    try:
        entities = gliner.predict_entities(
            extract_text,
            ENTITY_LABELS,
            threshold=GLINER_THRESHOLD,
        )
    except Exception as e:
        print(f"    GLiNER error on {sc.slide_id}: {e}")
        return []

    if not entities:
        return []

    # Collapse duplicate spans -- keep highest score per phrase
    best: Dict[str, float] = {}
    for ent in entities:
        phrase = ent["text"].lower().strip()
        score  = float(ent["score"])
        if len(phrase) < 3:
            continue
        if phrase not in best or score > best[phrase]:
            best[phrase] = score

    # Build Keyphrase objects
    kps: List[Keyphrase] = []
    for phrase, score in best.items():
        src   = _assign_source_type(phrase, sc)
        final = min(score + HEADING_BOOST, 1.0) if src == "heading" else score
        app   = _find_sentence(phrase, sc.clean_text)
        kps.append(Keyphrase(
            phrase=phrase,
            score=round(final, 4),
            source_type=src,
            slide_id=sc.slide_id,
            doc_id=sc.doc_id,
            appears_in=app,
        ))

    # Deduplicate near-synonyms, sort, cap
    kps = _deduplicate(kps)
    kps.sort(key=lambda k: k.score, reverse=True)
    return kps[:KEYPHRASE_MAX]

# -- Pedagogical slide filter -------------------------------------------------
def is_pedagogical_slide(sc) -> bool:
    """Return False for title/ToC slides and decorative/near-empty slides."""
    word_count = len(sc.clean_text.split())
    # Skip the first slide if it has a low word count (likely a title slide)
    if sc.page_number == 1 and word_count < 40:
        return False
    # Skip decorative/empty slides throughout the PDF
    if word_count < 8:
        return False
    return True

# -- Run on all pedagogical content slides ------------------------------------
print("\nExtracting keyphrases with GLiNER large from pedagogical slides ...")
keyphrases_by_slide: Dict[str, List[Keyphrase]] = {}
label_dist: Dict[str, int] = {}

for i, sc in enumerate(content_slides, 1):
    if not is_pedagogical_slide(sc):
        print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: SKIPPED (Administrative/Low-content, {len(sc.clean_text.split())} words)")
        keyphrases_by_slide[sc.slide_id] = []
        continue

    kps = extract_keyphrases(sc)
    keyphrases_by_slide[sc.slide_id] = kps

    # Collect raw entity labels for inspection
    raw_text = " ".join(sc.headings + sc.body_text + sc.bullets
                        + sc.table_cells + sc.captions).strip()
    if raw_text:
        try:
            raw_ents = gliner.predict_entities(
                raw_text, ENTITY_LABELS, threshold=GLINER_THRESHOLD
            )
            for e in raw_ents:
                label_dist[e["label"]] = label_dist.get(e["label"], 0) + 1
        except Exception:
            pass

    print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: {len(kps)} keyphrases")
    if kps:
        top3 = ", ".join(f"{k.phrase} ({k.score:.2f})" for k in kps[:3])
        print(f"             top-3: {top3}")

total_kps = sum(len(v) for v in keyphrases_by_slide.values())
print(f"\nStep 3 complete : {total_kps} keyphrases across {len(content_slides)} slides")
print(f"Avg per slide   : {total_kps/len(content_slides):.1f}")
print()
print("Entity label distribution (raw, before dedup):")
for lbl, cnt in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f"  {lbl:<38} {cnt}")

# -- Save ---------------------------------------------------------------------
STEP3_FILE = f"{STEM}_step3_keyphrases.json"
step3_out  = {sid: [asdict(k) for k in kps]
              for sid, kps in keyphrases_by_slide.items()}
with open(STEP3_FILE, "w", encoding="utf-8") as f:
    json.dump(step3_out, f, ensure_ascii=False, indent=2)
print(f"\nSaved  : {STEP3_FILE} ({Path(STEP3_FILE).stat().st_size:,} bytes)")


Loading GLiNER large-v2.1 on GPU ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

gliner_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


All models ready.

Extracting keyphrases with GLiNER large from pedagogical slides ...
  [ 1/28] slide_001: SKIPPED (Administrative/Low-content, 18 words)
  [ 2/28] slide_002: 1 keyphrases
             top-3: object oriented programming (0.62)
  [ 3/28] slide_003: 6 keyphrases
             top-3: queue (0.96), map (0.95), linked list (0.91)
  [ 4/28] slide_004: 2 keyphrases
             top-3: arrays (0.90), this array (0.58)
  [ 5/28] slide_005: 1 keyphrases
             top-3: expression (0.53)
  [ 6/28] slide_006: 6 keyphrases
             top-3: element type (0.76), array elements (0.67), object (0.53)
  [ 7/28] slide_007: 2 keyphrases
             top-3: an array of integers (0.46), int (0.38)
  [ 8/28] slide_008: 4 keyphrases
             top-3: arrayindexoutofboundsexception (0.84), bound checking (0.68), exception (0.66)
  [ 9/28] slide_009: 2 keyphrases
             top-3: length (1.00), scores (0.37)
  [10/28] slide_010: 7 keyphrases
             top-3: method (0.70), arrays 

In [10]:
# ── Step 3 inspection ─────────────────────────────────────────────────────
from IPython.display import Markdown, display

print(f"{'slide_id':<12}  {'#kps':>4}  top-5 keyphrases (phrase — score — source)")
print("-" * 100)
for sc in content_slides:
    kps = keyphrases_by_slide.get(sc.slide_id, [])
    top5 = ", ".join(
        f"{k.phrase} ({k.score:.2f}/{k.source_type[0]})"
        for k in kps[:5]
    )
    print(f"{sc.slide_id:<12}  {len(kps):>4}  {top5}")

slide_id      #kps  top-5 keyphrases (phrase — score — source)
----------------------------------------------------------------------------------------------------
slide_001        0  
slide_002        1  object oriented programming (0.62/h)
slide_003        6  queue (0.96/b), map (0.95/b), linked list (0.91/b), searching and sorting (0.86/b), list (0.74/b)
slide_004        2  arrays (0.90/h), this array (0.58/b)
slide_005        1  expression (0.53/b)
slide_006        6  element type (0.76/b), array elements (0.67/b), object (0.53/b), primitive type (0.44/b), an array of coin objects (0.43/b)
slide_007        2  an array of integers (0.46/b), int (0.38/b)
slide_008        4  arrayindexoutofboundsexception (0.84/b), bound checking (0.68/h), exception (0.66/b), java interpreter (0.58/b)
slide_009        2  length (1.00/h), scores (0.37/b)
slide_010        7  method (0.70/b), arrays (0.58/h), formal parameter (0.56/b), type (0.52/b), actual parameters (0.43/b)
slide_011        2  arrays 

In [11]:
# ── Download Step 3 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP3_FILE)
print(f"Downloading {STEP3_FILE} ...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 4 — LLM Triple Extraction

**Core novel contribution.** Sends (keyphrases + slide text) to Groq's Llama 3 and
validates each returned triple through **3 mandatory layers**:

| Layer | Check |
|-------|-------|
| 1 — Anchor | subject AND object must be in the keyphrase list; relation must be one of 8 types |
| 2 — Evidence | evidence string must be semantically close to slide text (SBERT cosine ≥ 0.75) |
| 3 — Confidence | LLM-reported confidence ≥ 0.70 |

Enter your **Groq API key** below. Free at [console.groq.com](https://console.groq.com).

In [12]:
import os
from getpass import getpass

# -- Groq API key -------------------------------------------------------------
GROQ_API_KEY = getpass("Paste your Groq API key (hidden): ")

if not GROQ_API_KEY.strip():
    raise ValueError("GROQ_API_KEY is empty -- cannot proceed with Step 4")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# -- Model config -------------------------------------------------------------
LLM_PRIMARY  = "llama-3.3-70b-versatile"
LLM_FALLBACK = "llama-3.1-8b-instant"

# -- Validation thresholds ----------------------------------------------------
TRIPLE_CONFIDENCE_THRESHOLD   = 0.70
EVIDENCE_SIMILARITY_THRESHOLD = 0.65   # FIX 4: was 0.75 (too strict, dropped valid triples)

print("Config ready.")
print(f"  Primary LLM            : {LLM_PRIMARY}")
print(f"  Fallback LLM           : {LLM_FALLBACK}")
print(f"  Confidence threshold   : {TRIPLE_CONFIDENCE_THRESHOLD}")
print(f"  Evidence sim threshold : {EVIDENCE_SIMILARITY_THRESHOLD}")


Paste your Groq API key (hidden): ··········
Config ready.
  Primary LLM            : llama-3.3-70b-versatile
  Fallback LLM           : llama-3.1-8b-instant
  Confidence threshold   : 0.7
  Evidence sim threshold : 0.65


In [13]:
import json, time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Dict, Optional

from sentence_transformers import SentenceTransformer, util as st_util
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

# -- Data model ---------------------------------------------------------------
@dataclass
class Triple:
    subject:    str
    relation:   str
    object:     str
    evidence:   str
    confidence: float
    slide_id:   str
    doc_id:     str
    source:     str = "extraction"

# -- FIX 5: Updated system prompt ---------------------------------------------
# Changes vs original:
#   - Explicit direction rule + example for each of the 8 relation types
#   - DIVERSITY REQUIREMENT: isPartOf capped at 40%, contrastedWith required
#     when slide title contains "vs" / "comparison" / "advantages"
#   - SLIDE HEADING RULE: block slide title from being used as triple object
SYSTEM_PROMPT = """You are a knowledge graph construction assistant for educational materials.

STRICT RULES -- violating any rule invalidates your entire response:
1. ONLY use concepts from the KEYPHRASE LIST as subject and object. Nothing else.
2. Every triple MUST include an EXACT sentence copied from the SLIDE TEXT as evidence.
3. If no supporting sentence exists in the slide text, omit that triple entirely.
4. Use ONLY these 8 relation types. Follow the DIRECTION rules exactly:

   isPrerequisiteOf   subject must be understood BEFORE object can be understood
                      e.g. "pointers --[isPrerequisiteOf]--> linked list"

   isDefinedAs        subject IS the concept being defined; object is the definition
                      e.g. "hashmap --[isDefinedAs]--> key-value store"
                      NEVER reverse: the concept comes first, the definition second

   isExampleOf        subject is a SPECIFIC INSTANCE of the broader concept object
                      e.g. "arraylist --[isExampleOf]--> collection"
                      e.g. "linkedlist --[isExampleOf]--> linked list data structure"

   contrastedWith     subject AND object are EXPLICITLY compared or contrasted
                      e.g. "hashmap --[contrastedWith]--> linkedlist"
                      USE THIS when the slide title contains "vs", "comparison", "advantages"
                      or when the slide text uses words like "unlike", "whereas", "compared to"

   appliedIn          subject concept IS USED IN or runs INSIDE object
                      e.g. "hash function --[appliedIn]--> hashmap"
                      e.g. "hashmap --[appliedIn]--> jvm"

   isPartOf           subject is a PHYSICAL or STRUCTURAL COMPONENT of object
                      e.g. "load factor --[isPartOf]--> hashmap parameters"
                      DO NOT use for: taxonomy membership, conceptual groupings,
                      or when one concept IS AN IMPLEMENTATION of another

   causeOf            subject DIRECTLY CAUSES or leads to object as a result
                      e.g. "rehashing --[causeOf]--> capacity increase"
                      e.g. "capacity x load factor --[causeOf]--> rehashing threshold"

   isGeneralizationOf subject is a BROADER CATEGORY that contains object as a member
                      e.g. "collection --[isGeneralizationOf]--> hashmap"

5. DIVERSITY REQUIREMENT:
   - Consider ALL 8 relation types before choosing one
   - Do NOT use isPartOf for more than 40% of your triples in this response
   - If the slide title contains "vs", "comparison", or "advantages", you MUST
     produce at least one contrastedWith triple
   - If the slide text contains "is an implementation of" or "is defined as",
     use isDefinedAs or isExampleOf -- not isPartOf
   - If the slide text contains "causes", "results in", or "leads to",
     use causeOf -- not isPartOf

6. SLIDE HEADING RULE:
   - The slide title (e.g. "Java pre- and post- Collections", "HashMap Class")
     is a TOPIC LABEL for the slide, not an educational concept
   - Do NOT use the full slide title string as a triple object
   - Use the specific concepts within the slide instead

7. Return ONLY a valid JSON array. No markdown, no explanation, no preamble.
8. If no valid triples found, return: []

CRITICAL CONSTRAINTS:
1. NO ADMINISTRATIVE ENTITIES: Do not extract relationships between names of people, professors, universities, departments, course codes, or dates.
2. PEDAGOGICAL ONLY: Only extract relationships that represent academic, domain-specific, or theoretical concepts taught in the material.
3. EMPTY FALLBACK: If the source text contains only a title slide, administrative info, a table of contents, or no clear academic facts, output an empty array []."""

USER_PROMPT = """KEYPHRASE LIST (subject and object MUST come from this list):
{keyphrases}

SLIDE TEXT:
{slide_text}

Extract all valid triples as JSON array. Each item:
{{
  "subject": "exact phrase from keyphrase list",
  "relation": "one of the 8 relation types",
  "object": "exact phrase from keyphrase list (different from subject)",
  "evidence": "exact sentence copied from slide text above",
  "confidence": 0.0 to 1.0
}}"""

# -- LLM clients --------------------------------------------------------------
_primary_llm  = None
_fallback_llm = None

def _get_primary() -> ChatGroq:
    global _primary_llm
    if _primary_llm is None:
        _primary_llm = ChatGroq(
            model=LLM_PRIMARY,
            temperature=0,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    return _primary_llm

def _get_fallback() -> ChatGroq:
    global _fallback_llm
    if _fallback_llm is None:
        _fallback_llm = ChatGroq(
            model=LLM_FALLBACK,
            temperature=0,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    return _fallback_llm

def _invoke_with_fallback(messages: list, max_retries: int = 3) -> Optional[str]:
    for attempt in range(1, max_retries + 1):
        try:
            return str(_get_primary().invoke(messages).content)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print(f"    Rate-limited (attempt {attempt}) -- trying fallback ...")
                try:
                    return str(_get_fallback().invoke(messages).content)
                except Exception as e2:
                    if "429" in str(e2) or "rate" in str(e2).lower():
                        print(f"    Fallback also rate-limited -- waiting 5s ...")
                        time.sleep(5)
                        continue
                    print(f"    Fallback error: {e2}")
                    return None
            print(f"    LLM error: {e}")
            return None
    return None

def _parse_llm_response(content: str) -> List[dict]:
    if not content:
        return []
    try:
        raw = json.loads(content)
    except json.JSONDecodeError:
        return []
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for v in raw.values():
            if isinstance(v, list):
                return v
    return []

# -- Load SBERT for evidence verification -------------------------------------
print("Loading all-mpnet-base-v2 for evidence verification ...")
sbert = SentenceTransformer("all-mpnet-base-v2")
print("SBERT ready.")

_VALID_RELATIONS = frozenset({
    "isPrerequisiteOf", "isDefinedAs", "isExampleOf", "contrastedWith",
    "appliedIn", "isPartOf", "causeOf", "isGeneralizationOf",
})

# -- 3-layer validator --------------------------------------------------------
def validate_triple(
    raw: dict,
    kp_set: set,
    slide_text: str,
    slide_id: str,
    doc_id: str,
) -> Optional[Triple]:
    subject  = str(raw.get("subject",   "")).lower().strip()
    obj      = str(raw.get("object",    "")).lower().strip()
    relation = str(raw.get("relation",  "")).strip()
    evidence = str(raw.get("evidence",  "")).strip()
    try:
        conf = float(raw.get("confidence", 0))
    except (TypeError, ValueError):
        conf = 0.0

    # Layer 1: anchor + schema
    if subject not in kp_set or obj not in kp_set:
        return None
    if subject == obj:
        return None
    if relation not in _VALID_RELATIONS:
        return None

    # Layer 2: evidence verification (FIX 4: threshold lowered to 0.65)
    if not evidence:
        return None
    ev_emb   = sbert.encode(evidence,   convert_to_tensor=True, show_progress_bar=False)
    text_emb = sbert.encode(slide_text, convert_to_tensor=True, show_progress_bar=False)
    sim = float(st_util.cos_sim(ev_emb, text_emb))
    if sim < EVIDENCE_SIMILARITY_THRESHOLD:
        return None

    # Layer 3: confidence threshold
    if conf < TRIPLE_CONFIDENCE_THRESHOLD:
        return None

    return Triple(
        subject=subject, relation=relation, object=obj,
        evidence=evidence, confidence=conf,
        slide_id=slide_id, doc_id=doc_id,
    )

# -- Run extraction -----------------------------------------------------------
print("\nExtracting triples (1 LLM call per slide) ...")
all_triples: List[Triple] = []
stats = {"slides": 0, "skipped": 0, "raw": 0, "passed": 0}

for i, sc in enumerate(content_slides, 1):
    kps = keyphrases_by_slide.get(sc.slide_id, [])
    if not kps or not sc.clean_text.strip():
        print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: skipped (no keyphrases or text)")
        stats["skipped"] += 1
        continue

    kp_list  = "\n".join(f"- {k.phrase}" for k in kps)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=USER_PROMPT.format(
            keyphrases=kp_list,
            slide_text=sc.clean_text,
        )),
    ]

    content_raw = _invoke_with_fallback(messages)
    raw_dicts   = _parse_llm_response(content_raw or "")
    stats["raw"] += len(raw_dicts)

    kp_set = {k.phrase.lower() for k in kps}
    slide_triples: List[Triple] = []
    for rd in raw_dicts:
        t = validate_triple(rd, kp_set, sc.clean_text, sc.slide_id, sc.doc_id)
        if t is not None:
            slide_triples.append(t)

    all_triples.extend(slide_triples)
    stats["slides"] += 1
    stats["passed"] += len(slide_triples)

    print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: "
          f"{len(raw_dicts)} raw  {len(slide_triples)} valid")

print(f"\nStep 4 complete:")
print(f"  Slides processed  : {stats['slides']} (skipped: {stats['skipped']})")
print(f"  Raw candidates    : {stats['raw']}")
if stats["raw"]:
    print(f"  Validated triples : {stats['passed']} "
          f"({100*stats['passed']/stats['raw']:.0f}% pass rate)")

# Relation distribution
rel_dist = Counter(t.relation for t in all_triples)
print("\nRelation distribution:")
for rel, cnt in sorted(rel_dist.items(), key=lambda x: -x[1]):
    bar = "#" * cnt
    print(f"  {rel:<22} {cnt:>3}  {bar}")

STEP4_FILE = f"{STEM}_step4_triples.json"
with open(STEP4_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(t) for t in all_triples], f, ensure_ascii=False, indent=2)
print(f"\nSaved  : {STEP4_FILE} ({Path(STEP4_FILE).stat().st_size:,} bytes)")


Loading all-mpnet-base-v2 for evidence verification ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT ready.

Extracting triples (1 LLM call per slide) ...
  [ 1/28] slide_001: skipped (no keyphrases or text)
  [ 2/28] slide_002: 0 raw  0 valid
  [ 3/28] slide_003: 6 raw  3 valid
  [ 4/28] slide_004: 1 raw  1 valid
  [ 5/28] slide_005: 0 raw  0 valid
  [ 6/28] slide_006: 4 raw  2 valid
  [ 7/28] slide_007: 0 raw  0 valid
  [ 8/28] slide_008: 3 raw  1 valid
  [ 9/28] slide_009: 0 raw  0 valid
  [10/28] slide_010: 7 raw  4 valid
  [11/28] slide_011: 2 raw  2 valid
  [12/28] slide_012: 6 raw  1 valid
  [13/28] slide_013: 3 raw  2 valid
  [14/28] slide_014: 3 raw  2 valid
  [15/28] slide_015: 2 raw  1 valid
  [16/28] slide_016: 0 raw  0 valid
  [17/28] slide_017: skipped (no keyphrases or text)
  [18/28] slide_018: 0 raw  0 valid
  [19/28] slide_019: skipped (no keyphrases or text)
  [20/28] slide_020: 7 raw  5 valid
  [21/28] slide_021: 8 raw  7 valid
  [22/28] slide_022: 5 raw  4 valid
  [23/28] slide_023: 1 raw  0 valid
  [24/28] slide_024: skipped (no keyphrases or text)
  [25/28

In [14]:
# ── Step 4 inspection ─────────────────────────────────────────────────────
from collections import Counter
from IPython.display import Markdown, display

rel_counts = Counter(t.relation for t in all_triples)

print(f"Total validated triples : {len(all_triples)}")
print(f"\nRelation type distribution:")
for rel, cnt in rel_counts.most_common():
    bar = "█" * cnt
    print(f"  {rel:<22} {cnt:>3}  {bar}")

print("\n" + "=" * 80)
print("Sample triples (first 10):")
for t in all_triples[:10]:
    display(Markdown(
        f"**[{t.slide_id}]** `{t.subject}` —*{t.relation}*→ `{t.object}`  "
        f"conf={t.confidence:.2f}  "
        f"\n> *{t.evidence[:120]}*"
    ))

Total validated triples : 38

Relation type distribution:
  isExampleOf             19  ███████████████████
  isPartOf                10  ██████████
  isGeneralizationOf       5  █████
  isDefinedAs              2  ██
  isPrerequisiteOf         1  █
  contrastedWith           1  █

Sample triples (first 10):


**[slide_003]** `linked list` —*isExampleOf*→ `list`  conf=0.80  
> *Summary Arrays List, Queue, Map Searching and Sorting Linked List*

**[slide_003]** `queue` —*isExampleOf*→ `list`  conf=0.80  
> *Summary Arrays List, Queue, Map Searching and Sorting Linked List*

**[slide_003]** `arrays` —*isExampleOf*→ `list`  conf=0.80  
> *Summary Arrays List, Queue, Map Searching and Sorting Linked List*

**[slide_004]** `this array` —*isExampleOf*→ `arrays`  conf=0.80  
> *This array holds 10 values that are indexed from 0 to 9*

...

**[slide_006]** `an array of coin objects` —*isExampleOf*→ `array elements`  conf=0.70  
> *We can create an array of integers, an array of characters, an array of String objects, an array of Coin objects, etc.*

**[slide_008]** `bound checking` —*isPrerequisiteOf*→ `exception`  conf=0.80  
> *If the array scores can hold 10 values, it can be indexed using only the numbers 0 to 9 If the value of count is 10, the*

**[slide_010]** `arrays` —*isExampleOf*→ `reference`  conf=0.80  
> *Like any other object, the reference to the array is passed, making the formal and actual parameters aliases of each oth*

**[slide_010]** `formal parameter` —*isExampleOf*→ `aliases`  conf=0.80  
> *Like any other object, the reference to the array is passed, making the formal and actual parameters aliases of each oth*

**[slide_010]** `actual parameters` —*isExampleOf*→ `aliases`  conf=0.80  
> *Like any other object, the reference to the array is passed, making the formal and actual parameters aliases of each oth*

In [15]:
# ── Download Step 4 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP4_FILE)
print(f"Downloading {STEP4_FILE} ...")

# ── Download all intermediate files at once (optional) ────────────────────
print("\nDownload all intermediate files:")
for fname in [STEP1_FILE, STEP2_FILE, STEP3_FILE, STEP4_FILE]:
    _files.download(fname)
    print(f"  {fname}")
print("Done — check your browser Downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download all intermediate files:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  test3_step3_keyphrases.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  test3_step4_triples.json
Done — check your browser Downloads folder.
